# Bab 11 — Feature Selection and Feature Engineering

Notebook pendamping Bab 11 Level 2.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

In [ ]:
import os
from pathlib import Path

# Pindah ke folder utama proyek agar "data/mall_customers.csv" ketemu.
if not Path("data").exists():
    os.chdir("../..")

## Langkah 1: Siapkan alat dan muat data

In [ ]:
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)

DATA_PATH = "data/mall_customers.csv"
df = pd.read_csv(DATA_PATH)

## Langkah 2: Membuang kolom yang bukan fitur

In [ ]:
print("Kolom sebelum:", df.columns.tolist())

df_fitur = df.drop(columns=["CustomerID"])

print("Kolom sesudah:", df_fitur.columns.tolist())

## Langkah 3: Mencari fitur yang berulang (redundan)

In [ ]:
angka = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
korelasi = df[angka].corr().round(2)
print(korelasi)

pasangan_kuat = []
for i, a in enumerate(angka):
    for b in angka[i + 1:]:
        if abs(korelasi.loc[a, b]) >= 0.8:
            pasangan_kuat.append((a, b))

print("\nPasangan dengan |korelasi| >= 0.8:", pasangan_kuat)

## Langkah 4: Membuat fitur baru

In [ ]:
df_baru = df_fitur.copy()
df_baru["Spend_Income_Ratio"] = (
    df_baru["Spending Score (1-100)"] / df_baru["Annual Income (k$)"]
)

print(df_baru.head())

## Langkah 5: Menilai apakah fitur baru layak dipakai

In [ ]:
rasio = df_baru["Spend_Income_Ratio"]

print("Ringkasan fitur baru:")
print(rasio.describe().round(2))
print("\nSkewness fitur baru:", round(rasio.skew(), 2))

print("\nKorelasi fitur baru dengan fitur asal:")
print(df_baru[angka + ["Spend_Income_Ratio"]].corr().loc["Spend_Income_Ratio"].round(2))

## Langkah 6: Menyusun kandidat himpunan fitur

In [ ]:
X_dua = df[["Annual Income (k$)", "Spending Score (1-100)"]]
X_tiga = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]]
X_empat = pd.get_dummies(
    df[["Gender", "Age", "Annual Income (k$)", "Spending Score (1-100)"]],
    columns=["Gender"],
    dtype=int,
)

for nama, X in [("2 fitur", X_dua), ("3 fitur", X_tiga), ("3 fitur + Gender", X_empat)]:
    print(f"{nama:18s}: bentuk {X.shape}, kolom {X.columns.tolist()}")